# 探索 EDD:golden set + diagnose(自己玩)

亲手跑通 **trace → eval → diagnose** 的中段:

1. 看 golden set 的构成(44 条)
2. 跑单条,看答案 / 引证 / 拒答
3. 把一批分类(diagnose)
4. **关键实验**:把 `wrongly_refused` 的题改用 **agentic 多跳** 重跑,看能救回多少
5. 看一条 trace

随便改参数、随便加 cell。每跑一条可答题 = 一次 Claude 调用(几分钱)。

In [ ]:
import os
from pathlib import Path

# 让 cwd = 仓库根目录(这样 data/ 和 evals/ 路径都对)
while not Path("pyproject.toml").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
print("cwd:", Path.cwd())

import json
import pandas as pd

from policy_copilot.agent import answer
from policy_copilot.tool_agent import answer_agentic
from policy_copilot.index import load_index

golden = [json.loads(l) for l in Path("evals/golden.jsonl").read_text().splitlines() if l.strip()]
print(len(golden), "golden cases")

# 索引只加载一次(嵌入模型有点慢,等十几秒)
index, chunks = load_index()
print("index ready")

## 1. golden set 长什么样

40 条可答(来自 FinanceBench,带金标答案+证据)+ 4 条手写拒答。

In [ ]:
g = pd.DataFrame(golden)
print("types:", g["type"].value_counts().to_dict())
print("by doc:", g[g.type == "answerable"]["doc_name"].value_counts().to_dict())
g[["id", "type", "doc_name", "question"]].head(12)

## 2. 跑单条(改 `idx` 试不同题)

前面几条多是**复杂分析题**(需要综合/计算),正好看严格 grounding 的反应。

In [ ]:
idx = 0  # ← 改这个下标试别的题
case = golden[idx]
print("Q   :", case["question"])
print("gold:", case.get("gold_answer", "(refusal case)")[:200])

res = answer(case["question"], index, chunks)
print("\nrefused  :", res.refused)
print("answer   :", res.text[:300])
print("citations:", res.citations)

## 3. diagnose:把一批分类

这就是 `evals/run_eval.py` 的核心 —— 跑一遍、归类、看失败模式。`N` 调大更全但更慢。

In [ ]:
def classify(case, res):
    if case["type"] == "refusal":
        return "ok" if res.refused else "should_have_refused"
    if res.refused:
        return "wrongly_refused"
    exp = case.get("expected_contains") or []
    if exp and not any(e in res.text for e in exp):
        return "missing_expected"
    if not res.citations:
        return "no_citation"
    return "ok"


N = 8  # ← 跑多少条可答题(每条一次 Claude 调用)
sample = [c for c in golden if c["type"] == "answerable"][:N]
sample += [c for c in golden if c["type"] == "refusal"]

rows = []
for c in sample:
    r = answer(c["question"], index, chunks)
    rows.append({"id": c["id"], "type": c["type"], "cat": classify(c, r), "q": c["question"][:50]})
diag = pd.DataFrame(rows)
print(diag["cat"].value_counts().to_dict())
diag

## 4. ⭐ 关键实验:`wrongly_refused` 改走 agentic 多跳

**假设**:这些题被拒,是因为单跳 + 严格 grounding 应付不了多步分析。

**验证**:同一批题用 `answer_agentic`(Claude 自己决定多次检索)重跑,看 `refused` 是否变 False、是否带引证+通过校验。这就闭合了 **diagnose → fix → re-measure** 循环。

In [ ]:
failed = diag[diag.cat == "wrongly_refused"]["id"].tolist()
print("wrongly_refused:", failed)

comp = []
for cid in failed:
    c = next(x for x in golden if x["id"] == cid)
    a = answer_agentic(c["question"], index, chunks)  # 多跳
    comp.append({
        "id": cid,
        "q": c["question"][:45],
        "agentic_refused": a.refused,
        "n_search": len(a.tool_calls),
        "verified": a.verdict.citations_resolve and a.verdict.numbers_verbatim,
        "answer": a.text[:100],
    })
comp_df = pd.DataFrame(comp)
recovered = 0 if comp_df.empty else int((~comp_df["agentic_refused"]).sum())
print(f"救回(不再拒答): {recovered} / {len(failed)}")
comp_df

## 5. 看一条 trace

每次回答都写进 `data/traces/traces.jsonl`;若开了 Langfuse,去 http://localhost:3000 看嵌套 trace + 成本。

In [ ]:
from policy_copilot.tracing import load_traces

tr = pd.DataFrame(load_traces())
print(len(tr), "traces")
cols = [c for c in ["mode", "question", "refused", "input_tokens", "output_tokens", "cost_usd", "error"] if c in tr.columns]
tr[cols].tail(8)

## 6. 自由发挥

几个玩法:
- 把 `N` 调到 40,跑全量可答题,看完整失败分布。
- 对 `missing_expected` 的题,看金标数字是不是**推导值**(原文没逐字)——这是"范围决策"。
- 把你发现的新失败,按 `golden.jsonl` 格式补成新 case(diagnose → 新 golden)。
- 对比 `answer` vs `answer_agentic` 在同一题上的 `cost_usd` / `n_search`。

In [ ]:
# 你的实验区:随便写
q = "What were Boeing's total revenues in 2022?"
r = answer_agentic(q, index, chunks)
print(r.text[:300])
print("verified:", r.verdict, "| searches:", len(r.tool_calls), "| cost $", r.cost_usd)